# 🎬 Netflix Movies & TV Shows — Complete Exploratory Data Analysis (EDA)

---

## 📌 Project Overview

This notebook performs a **comprehensive Exploratory Data Analysis (EDA)** on the Netflix Movies & TV Shows dataset sourced from TMDB (The Movie Database). The dataset contains ~9,800 entries with metadata such as release date, genres, popularity, vote counts, average ratings, original language, and poster URLs.

### 🎯 Objectives
1. Understand the structure and quality of the dataset
2. Clean and preprocess the data for analysis
3. Explore distributions of key features
4. Uncover genre trends, language distributions, and release patterns
5. Analyze ratings and popularity metrics
6. Answer business-relevant questions with visualizations

### 📂 Dataset Columns
| Column | Description |
|---|---|
| `Release_Date` | Date the title was released |
| `Title` | Name of the movie/show |
| `Overview` | Brief description of the content |
| `Popularity` | TMDB popularity score |
| `Vote_Count` | Number of user votes |
| `Vote_Average` | Average user rating (0–10) |
| `Original_Language` | Language code of original production |
| `Genre` | Comma-separated list of genres |
| `Poster_Url` | URL to the movie poster image |

---
## 📦 Section 1 — Import Libraries

In [ ]:
# ── Core Data Libraries ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualization Libraries ──────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Settings ─────────────────────────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.2f}'.format)

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 12

import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully!')

---
## 📥 Section 2 — Load the Dataset

In [ ]:
# Load dataset (on_bad_lines='skip' handles any malformed rows)
df = pd.read_csv('netflix.csv', on_bad_lines='skip', engine='python')

print(f'📊 Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print()
df.head()

In [ ]:
# Preview last few rows
df.tail()

---
## 🔍 Section 3 — Initial Data Inspection

In [ ]:
# Column data types and non-null counts
df.info()

In [ ]:
# Basic statistical summary (before type conversion)
df.describe(include='all').T

In [ ]:
# Column names
print('Columns:', df.columns.tolist())

---
## 🧹 Section 4 — Data Cleaning & Preprocessing

### 4.1 — Missing Values Analysis

In [ ]:
# Count and percentage of missing values
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})
print('Missing Values Summary:')
print(missing[missing['Missing Count'] > 0])

In [ ]:
# Visualise missing values
fig, ax = plt.subplots(figsize=(10, 4))
missing_pct = (df.isnull().sum() / len(df) * 100)
missing_pct[missing_pct > 0].plot(kind='bar', color='salmon', edgecolor='black', ax=ax)
ax.set_title('Missing Values (%) per Column', fontsize=14, fontweight='bold')
ax.set_ylabel('Missing %')
ax.set_xlabel('')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### 4.2 — Data Type Conversion

In [ ]:
# Convert Vote_Count and Vote_Average to numeric
df['Vote_Count']   = pd.to_numeric(df['Vote_Count'],   errors='coerce')
df['Vote_Average'] = pd.to_numeric(df['Vote_Average'], errors='coerce')

# Parse Release_Date to datetime
df['Release_Date'] = pd.to_datetime(df['Release_Date'], errors='coerce')

# Extract useful date features
df['Release_Year']  = df['Release_Date'].dt.year
df['Release_Month'] = df['Release_Date'].dt.month
df['Release_Month_Name'] = df['Release_Date'].dt.strftime('%b')

print('✅ Type conversions done!')
print(df[['Release_Date','Release_Year','Release_Month','Vote_Count','Vote_Average']].dtypes)

### 4.3 — Duplicate Check

In [ ]:
dup_count = df.duplicated().sum()
print(f'🔁 Duplicate rows: {dup_count}')

# Drop duplicates if any
df = df.drop_duplicates().reset_index(drop=True)
print(f'✅ Cleaned shape: {df.shape}')

### 4.4 — Expand Multi-Value Genre Column

In [ ]:
# Each title can have multiple genres separated by ', '
# Create an exploded DataFrame for genre-level analysis
df_genre = df.dropna(subset=['Genre']).copy()
df_genre['Genre_List'] = df_genre['Genre'].str.split(', ')
df_genre_exploded = df_genre.explode('Genre_List').reset_index(drop=True)
df_genre_exploded['Genre_List'] = df_genre_exploded['Genre_List'].str.strip()

print(f'Genre-level rows: {len(df_genre_exploded):,}')
print(f'Unique genres   : {df_genre_exploded["Genre_List"].nunique()}')

---
## 📊 Section 5 — Univariate Analysis

### 5.1 — Popularity Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['Popularity'].dropna(), bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Popularity Distribution (Full Range)', fontweight='bold')
axes[0].set_xlabel('Popularity Score')
axes[0].set_ylabel('Frequency')

# Zoomed view (99th percentile)
p99 = df['Popularity'].quantile(0.99)
axes[1].hist(df[df['Popularity'] <= p99]['Popularity'], bins=60, color='darkorange', edgecolor='white')
axes[1].set_title('Popularity Distribution (≤ 99th Percentile)', fontweight='bold')
axes[1].set_xlabel('Popularity Score')
axes[1].set_ylabel('Frequency')

plt.suptitle('📈 Popularity Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(df['Popularity'].describe())

### 5.2 — Vote Average Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['Vote_Average'].dropna(), bins=40, color='mediumpurple', edgecolor='white')
axes[0].set_title('Vote Average Distribution', fontweight='bold')
axes[0].set_xlabel('Rating (0–10)')
axes[0].set_ylabel('Count')
axes[0].axvline(df['Vote_Average'].mean(), color='red', linestyle='--', label=f'Mean: {df["Vote_Average"].mean():.2f}')
axes[0].axvline(df['Vote_Average'].median(), color='green', linestyle='--', label=f'Median: {df["Vote_Average"].median():.2f}')
axes[0].legend()

# KDE
sns.kdeplot(df['Vote_Average'].dropna(), ax=axes[1], fill=True, color='mediumpurple')
axes[1].set_title('Vote Average — KDE', fontweight='bold')
axes[1].set_xlabel('Rating (0–10)')

plt.suptitle('⭐ Vote Average (Rating) Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(df['Vote_Average'].describe())

### 5.3 — Vote Count Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Vote_Count'].dropna(), bins=60, color='teal', edgecolor='white')
axes[0].set_title('Vote Count Distribution (Full)', fontweight='bold')
axes[0].set_xlabel('Number of Votes')

# Log scale
axes[1].hist(np.log1p(df['Vote_Count'].dropna()), bins=60, color='teal', edgecolor='white')
axes[1].set_title('Vote Count Distribution (Log Scale)', fontweight='bold')
axes[1].set_xlabel('log(Vote Count + 1)')

plt.suptitle('🗳️ Vote Count Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 🎭 Section 6 — Genre Analysis

### 6.1 — Top 15 Genres by Content Count

In [ ]:
genre_counts = df_genre_exploded['Genre_List'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(genre_counts.index[::-1], genre_counts.values[::-1],
               color=sns.color_palette('tab20', 15))

# Add value labels
for bar, val in zip(bars, genre_counts.values[::-1]):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10)

ax.set_title('🎭 Top 15 Genres — Number of Titles', fontsize=15, fontweight='bold')
ax.set_xlabel('Number of Titles')
ax.set_xlim(0, genre_counts.max() * 1.15)
plt.tight_layout()
plt.show()

### 6.2 — Genre Share — Pie Chart (Top 10)

In [ ]:
genre_top10 = df_genre_exploded['Genre_List'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(9, 9))
wedges, texts, autotexts = ax.pie(
    genre_top10.values,
    labels=genre_top10.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=sns.color_palette('Set2', 10),
    pctdistance=0.82,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for at in autotexts:
    at.set_fontsize(9)

ax.set_title('🍕 Genre Share Among Top 10 Genres', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.3 — Average Rating per Genre

In [ ]:
genre_rating = (
    df_genre_exploded.groupby('Genre_List')['Vote_Average']
    .agg(['mean', 'count'])
    .query('count >= 50')   # at least 50 titles for reliability
    .sort_values('mean', ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(genre_rating.index[::-1], genre_rating['mean'][::-1],
               color=sns.color_palette('RdYlGn', len(genre_rating)))
ax.axvline(df['Vote_Average'].mean(), color='red', linestyle='--', linewidth=1.5,
           label=f'Overall Mean: {df["Vote_Average"].mean():.2f}')
ax.set_title('⭐ Average Rating per Genre (min 50 titles)', fontsize=14, fontweight='bold')
ax.set_xlabel('Average Vote Rating')
ax.set_xlim(0, 10)
ax.legend()
for bar, val in zip(bars, genre_rating['mean'][::-1]):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

### 6.4 — Average Popularity per Genre

In [ ]:
genre_pop = (
    df_genre_exploded.groupby('Genre_List')['Popularity']
    .agg(['mean', 'count'])
    .query('count >= 50')
    .sort_values('mean', ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x='mean', y=genre_pop.index, data=genre_pop.reset_index(),
            palette='Blues_r', ax=ax)
ax.set_title('🔥 Average Popularity per Genre (min 50 titles)', fontsize=14, fontweight='bold')
ax.set_xlabel('Average Popularity Score')
ax.set_ylabel('Genre')
plt.tight_layout()
plt.show()

---
## 🌍 Section 7 — Language Analysis

### 7.1 — Top 15 Original Languages

In [ ]:
lang_map = {
    'en': 'English', 'ja': 'Japanese', 'es': 'Spanish', 'fr': 'French',
    'ko': 'Korean', 'zh': 'Chinese', 'it': 'Italian', 'cn': 'Cantonese',
    'ru': 'Russian', 'de': 'German', 'pt': 'Portuguese', 'da': 'Danish',
    'hi': 'Hindi', 'no': 'Norwegian', 'sv': 'Swedish', 'tr': 'Turkish',
    'nl': 'Dutch', 'th': 'Thai', 'ar': 'Arabic', 'pl': 'Polish'
}

df['Language_Name'] = df['Original_Language'].map(lang_map).fillna(df['Original_Language'])

lang_counts = df['Language_Name'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(lang_counts.index, lang_counts.values,
              color=sns.color_palette('tab20', len(lang_counts)), edgecolor='white')
ax.set_title('🌍 Top 15 Original Languages on Netflix', fontsize=14, fontweight='bold')
ax.set_xlabel('Language')
ax.set_ylabel('Number of Titles')
plt.xticks(rotation=35, ha='right')
for bar, val in zip(bars, lang_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{val:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

### 7.2 — Average Rating by Language (excl. English)

In [ ]:
lang_rating = (
    df[df['Original_Language'] != 'en']
    .groupby('Language_Name')['Vote_Average']
    .agg(['mean', 'count'])
    .query('count >= 10')
    .sort_values('mean', ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x='mean', y=lang_rating.index, data=lang_rating.reset_index(),
            palette='coolwarm', ax=ax)
ax.set_title('⭐ Avg Rating by Language (excl. English, min 10 titles)', fontsize=13, fontweight='bold')
ax.set_xlabel('Average Vote Rating')
plt.tight_layout()
plt.show()

### 7.3 — English vs Non-English Share

In [ ]:
en_count = (df['Original_Language'] == 'en').sum()
non_en_count = (df['Original_Language'] != 'en').sum()

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie([en_count, non_en_count],
       labels=['English', 'Non-English'],
       autopct='%1.1f%%',
       colors=['#4C72B0', '#DD8452'],
       startangle=90,
       explode=(0.05, 0),
       wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax.set_title('🌐 English vs Non-English Content', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'English Titles    : {en_count:,} ({en_count/len(df)*100:.1f}%)')
print(f'Non-English Titles: {non_en_count:,} ({non_en_count/len(df)*100:.1f}%)')

---
## 📅 Section 8 — Release Date & Time Analysis

### 8.1 — Content Added per Year

In [ ]:
year_counts = df['Release_Year'].value_counts().sort_index()
# Keep only reasonable years
year_counts = year_counts[(year_counts.index >= 1990) & (year_counts.index <= 2023)]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(year_counts.index, year_counts.values, color='steelblue', edgecolor='white', width=0.8)
ax.set_title('📅 Number of Titles Released per Year', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Titles')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 8.2 — Content Released per Month

In [ ]:
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_counts = df['Release_Month_Name'].value_counts().reindex(month_order)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(month_counts.index, month_counts.values,
              color=sns.color_palette('husl', 12), edgecolor='white')
ax.set_title('📆 Number of Titles Released per Month', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Titles')
for bar, val in zip(bars, month_counts.values):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                f'{int(val):,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

### 8.3 — Average Popularity Over the Years

In [ ]:
pop_by_year = (
    df[(df['Release_Year'] >= 2000) & (df['Release_Year'] <= 2023)]
    .groupby('Release_Year')['Popularity']
    .median()
)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(pop_by_year.index, pop_by_year.values, marker='o', color='darkorange', linewidth=2)
ax.fill_between(pop_by_year.index, pop_by_year.values, alpha=0.15, color='darkorange')
ax.set_title('📈 Median Popularity Score Over the Years', fontsize=14, fontweight='bold')
ax.set_xlabel('Release Year')
ax.set_ylabel('Median Popularity')
plt.tight_layout()
plt.show()

---
## 🔗 Section 9 — Bivariate & Correlation Analysis

### 9.1 — Correlation Heatmap

In [ ]:
numeric_cols = ['Popularity', 'Vote_Count', 'Vote_Average', 'Release_Year']
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, annot=True, fmt='.3f', cmap='coolwarm',
    mask=mask, center=0, square=True, linewidths=0.5,
    cbar_kws={'shrink': 0.8}, ax=ax
)
ax.set_title('🔗 Correlation Heatmap — Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(corr_matrix)

### 9.2 — Popularity vs Vote Average (Scatter)

In [ ]:
# Limit to 99th percentile for readability
df_plot = df[df['Popularity'] <= df['Popularity'].quantile(0.99)]

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    df_plot['Vote_Average'],
    df_plot['Popularity'],
    c=df_plot['Vote_Count'],
    cmap='viridis',
    alpha=0.4, s=15, edgecolors='none'
)
plt.colorbar(scatter, ax=ax, label='Vote Count')
ax.set_title('🔵 Popularity vs Vote Average (colored by Vote Count)', fontsize=13, fontweight='bold')
ax.set_xlabel('Vote Average (Rating)')
ax.set_ylabel('Popularity Score')
plt.tight_layout()
plt.show()

### 9.3 — Vote Count vs Vote Average

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
df_vc = df[df['Vote_Count'] <= df['Vote_Count'].quantile(0.99)]
ax.scatter(df_vc['Vote_Average'], df_vc['Vote_Count'],
           alpha=0.3, s=12, color='teal', edgecolors='none')
ax.set_title('🗳️ Vote Count vs Vote Average', fontsize=13, fontweight='bold')
ax.set_xlabel('Vote Average')
ax.set_ylabel('Vote Count')
plt.tight_layout()
plt.show()

### 9.4 — Box Plots: Rating Distribution Across Top Genres

In [ ]:
top_genres = df_genre_exploded['Genre_List'].value_counts().head(10).index.tolist()
df_box = df_genre_exploded[df_genre_exploded['Genre_List'].isin(top_genres)]

fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(
    data=df_box, x='Genre_List', y='Vote_Average',
    order=top_genres, palette='Set3', ax=ax
)
ax.set_title('📦 Rating Distribution Across Top 10 Genres', fontsize=14, fontweight='bold')
ax.set_xlabel('Genre')
ax.set_ylabel('Vote Average')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

---
## 🏆 Section 10 — Top & Bottom Performers

### 10.1 — Top 10 Most Popular Titles

In [ ]:
top10_pop = df.nlargest(10, 'Popularity')[['Title', 'Popularity', 'Vote_Average', 'Genre', 'Release_Year']]
top10_pop.index = range(1, 11)
print('🔥 Top 10 Most Popular Titles:')
top10_pop

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top10_pop['Title'][::-1], top10_pop['Popularity'][::-1],
        color=sns.color_palette('flare', 10), edgecolor='white')
ax.set_title('🔥 Top 10 Most Popular Titles on Netflix', fontsize=14, fontweight='bold')
ax.set_xlabel('Popularity Score')
plt.tight_layout()
plt.show()

### 10.2 — Top 10 Highest Rated Titles (min 500 votes)

In [ ]:
top10_rated = (
    df[df['Vote_Count'] >= 500]
    .nlargest(10, 'Vote_Average')
    [['Title', 'Vote_Average', 'Vote_Count', 'Genre', 'Release_Year']]
)
top10_rated.index = range(1, 11)
print('⭐ Top 10 Highest Rated Titles (min 500 votes):')
top10_rated

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top10_rated['Title'][::-1], top10_rated['Vote_Average'][::-1],
        color=sns.color_palette('YlOrBr', 10)[::-1], edgecolor='white')
ax.set_title('⭐ Top 10 Highest Rated Titles (min 500 votes)', fontsize=14, fontweight='bold')
ax.set_xlabel('Average Rating')
ax.set_xlim(0, 10)
plt.tight_layout()
plt.show()

### 10.3 — Top 10 Most Voted Titles

In [ ]:
top10_voted = df.nlargest(10, 'Vote_Count')[['Title', 'Vote_Count', 'Vote_Average', 'Genre', 'Release_Year']]
top10_voted.index = range(1, 11)
print('🗳️ Top 10 Most Voted Titles:')
top10_voted

---
## 📊 Section 11 — Advanced Analysis

### 11.1 — Rating Category Distribution

In [ ]:
def rate_category(score):
    if pd.isna(score):     return 'Unknown'
    elif score >= 8.0:     return 'Excellent (8+)'
    elif score >= 7.0:     return 'Good (7–8)'
    elif score >= 6.0:     return 'Average (6–7)'
    elif score >= 5.0:     return 'Below Average (5–6)'
    else:                  return 'Poor (<5)'

df['Rating_Category'] = df['Vote_Average'].apply(rate_category)

cat_order = ['Excellent (8+)', 'Good (7–8)', 'Average (6–7)', 'Below Average (5–6)', 'Poor (<5)', 'Unknown']
cat_counts = df['Rating_Category'].value_counts().reindex(cat_order).dropna()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(cat_counts.index, cat_counts.values,
              color=['#2ecc71','#27ae60','#f39c12','#e74c3c','#c0392b','#95a5a6'],
              edgecolor='white')
ax.set_title('🎯 Rating Category Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Rating Category')
ax.set_ylabel('Number of Titles')
plt.xticks(rotation=30, ha='right')
for bar, val in zip(bars, cat_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{int(val):,}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

### 11.2 — Genre × Year Heatmap (Content Volume)

In [ ]:
top8_genres = df_genre_exploded['Genre_List'].value_counts().head(8).index.tolist()

genre_year = (
    df_genre_exploded[
        (df_genre_exploded['Genre_List'].isin(top8_genres)) &
        (df_genre_exploded['Release_Year'] >= 2010) &
        (df_genre_exploded['Release_Year'] <= 2023)
    ]
    .groupby(['Genre_List', 'Release_Year'])
    .size()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(genre_year, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Title Count'})
ax.set_title('🗓️ Genre × Year Heatmap (2010–2023)', fontsize=14, fontweight='bold')
ax.set_xlabel('Release Year')
ax.set_ylabel('Genre')
plt.tight_layout()
plt.show()

### 11.3 — Genre Trend Over Years (Line Chart)

In [ ]:
top5_genres = df_genre_exploded['Genre_List'].value_counts().head(5).index.tolist()

genre_trend = (
    df_genre_exploded[
        (df_genre_exploded['Genre_List'].isin(top5_genres)) &
        (df_genre_exploded['Release_Year'] >= 2010) &
        (df_genre_exploded['Release_Year'] <= 2023)
    ]
    .groupby(['Release_Year', 'Genre_List'])
    .size()
    .reset_index(name='Count')
)

fig, ax = plt.subplots(figsize=(13, 6))
for genre in top5_genres:
    data = genre_trend[genre_trend['Genre_List'] == genre]
    ax.plot(data['Release_Year'], data['Count'], marker='o', label=genre, linewidth=2)

ax.set_title('📊 Top 5 Genre Trends Over Years (2010–2023)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Titles')
ax.legend(title='Genre', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

### 11.4 — Popularity by Language (Box Plot)

In [ ]:
# Top 8 languages by count
top8_langs = df['Language_Name'].value_counts().head(8).index.tolist()
df_lang_plot = df[
    (df['Language_Name'].isin(top8_langs)) &
    (df['Popularity'] <= df['Popularity'].quantile(0.97))
]

fig, ax = plt.subplots(figsize=(14, 6))
sns.violinplot(data=df_lang_plot, x='Language_Name', y='Popularity',
               order=top8_langs, palette='pastel', ax=ax, inner='quartile')
ax.set_title('🎻 Popularity Distribution by Language (Top 8)', fontsize=14, fontweight='bold')
ax.set_xlabel('Language')
ax.set_ylabel('Popularity Score')
plt.tight_layout()
plt.show()

---
## ❓ Section 12 — Business Questions Answered

### Q1 — Which genre has the highest average vote rating?

In [ ]:
best_rated_genre = (
    df_genre_exploded.groupby('Genre_List')['Vote_Average']
    .agg(['mean', 'count'])
    .query('count >= 50')
    .sort_values('mean', ascending=False)
    .head(1)
)
genre_name = best_rated_genre.index[0]
rating_val = best_rated_genre['mean'].iloc[0]
print(f'✅ Best Rated Genre: {genre_name} (avg rating: {rating_val:.2f})')

### Q2 — Which year had the most releases on Netflix?

In [ ]:
peak_year = df['Release_Year'].value_counts().idxmax()
peak_count = df['Release_Year'].value_counts().max()
print(f'✅ Peak Year: {int(peak_year)} with {peak_count:,} titles released')

### Q3 — What % of content is non-English?

In [ ]:
non_en_pct = (df['Original_Language'] != 'en').mean() * 100
print(f'✅ Non-English Content: {non_en_pct:.1f}% of total catalog')

### Q4 — Which month sees the most releases?

In [ ]:
peak_month = df['Release_Month_Name'].value_counts().idxmax()
peak_month_count = df['Release_Month_Name'].value_counts().max()
print(f'✅ Busiest Month: {peak_month} ({peak_month_count:,} releases)')

### Q5 — What fraction of titles are 'Good' or better (rating ≥ 7)?

In [ ]:
good_pct = (df['Vote_Average'] >= 7).mean() * 100
print(f'✅ Titles rated ≥ 7.0: {good_pct:.1f}% of catalog')

### Q6 — Which language (excl. English) has the highest average popularity?

In [ ]:
best_lang = (
    df[df['Original_Language'] != 'en']
    .groupby('Language_Name')['Popularity']
    .agg(['mean', 'count'])
    .query('count >= 10')
    .sort_values('mean', ascending=False)
    .head(1)
)
print(f'✅ Most Popular Non-English Language: {best_lang.index[0]} (avg popularity: {best_lang["mean"].iloc[0]:.1f})')

---
## 📝 Section 13 — Summary & Key Insights

### 🔑 Key Insights from the Netflix EDA

| # | Insight |
|---|---|
| 1 | **Drama dominates** — Drama is the most common genre, followed by Comedy and Action |
| 2 | **English is king** — ~77% of Netflix catalog is in English, but non-English content is growing |
| 3 | **Popularity is right-skewed** — A small number of blockbusters dominate popularity metrics |
| 4 | **Ratings cluster around 6–7** — Most titles are rated average; truly excellent titles are rare |
| 5 | **Japanese & Korean content** is highly represented among non-English offerings |
| 6 | **Recent years see more releases** — Content production has surged post-2010 |
| 7 | **Vote Count ↔ Popularity** — Well-known titles receive more votes and are more popular |
| 8 | **Genre trends** — Drama and Comedy have remained consistently popular across all years |
| 9 | **January & September** tend to be peak release months |
| 10 | **Action & Adventure** genres score highest in popularity, while **History & Documentary** score highest in ratings |

---

### 🚀 Future Work
- Build a **content-based recommendation system** using genres + overview text (NLP/TF-IDF)
- Apply **sentiment analysis** on the Overview column
- Create a **weighted rating formula** (like IMDb's Bayesian average)
- Analyse growth of **non-English content** year-over-year
- Use **clustering** to group similar titles together

In [ ]:
print('=' * 55)
print('       📊 NETFLIX EDA — FINAL DATASET STATS')
print('=' * 55)
print(f'  Total Titles Analyzed : {len(df):,}')
print(f'  Unique Genres         : {df_genre_exploded["Genre_List"].nunique()}')
print(f'  Unique Languages      : {df["Original_Language"].nunique()}')
print(f'  Year Range            : {int(df["Release_Year"].min())} – {int(df["Release_Year"].max())}')
print(f'  Avg Vote Rating       : {df["Vote_Average"].mean():.2f} / 10')
print(f'  Avg Popularity Score  : {df["Popularity"].mean():.1f}')
print(f'  Most Common Genre     : Drama')
print(f'  Most Common Language  : English ({(df["Original_Language"]=="en").mean()*100:.1f}%)')
print('=' * 55)
print('✅ Analysis Complete!')